# Earthdata catalog explorer

Explore the bundled NASA Earthdata catalog **without any network or credentials** — it ships as package data. The catalog spans nine DAACs reached through one Earthdata Login. See [Usage](../../reference/earthdata/usage.md) for the download walkthrough.

## Setup

The only import we need is the `Catalog` loader. Constructing it parses the package-data YAML into the curated `datasets` map plus the `daacs` provider registry — no network call.

In [ ]:
from collections import Counter

from earthlens.earthdata import Catalog

cat = Catalog()

### Catalog at a glance

The catalog carries three things: the curated `datasets` we model in detail, the `daacs` (CMR providers) they live under, and the much larger informational `available_datasets` index of every reachable id.

In [ ]:
print(f'curated datasets:        {len(cat.datasets)}')
print(f'CMR providers (DAACs):   {len(cat.daacs)}')
print(f'available_datasets index: {len(cat.available_datasets)}')

## Coverage by DAAC

Counting curated datasets per `daac` shows which archives are best represented — LP DAAC and GES DISC lead here.

In [ ]:
by_daac = Counter(ds.daac for ds in cat.datasets.values())
for daac, n in sorted(by_daac.items()):
    print(f'  {daac:12s} {n}')

## The defining property: output kind is per dataset

Unlike every other earthlens backend, Earthdata datasets are not all the same shape — gridded **raster**, point/profile **vector**, or **tabular**. Grouping by `output_kind` shows the split across the curated set.

In [ ]:
by_kind = Counter(ds.output_kind for ds in cat.datasets.values())
for kind, n in by_kind.most_common():
    print(f'  {kind:8s} {n}')

## Resolve a dataset and its DAAC

`get_dataset()` returns the modelled row for a concrete id; `get_daac()` then looks up the provider record (here to read its `cloud_region` for direct S3 access).

In [ ]:
ds = cat.get_dataset('GPM_3IMERGHHL_07')
print('short_name :', ds.short_name)
print('provider   :', ds.provider)
print('output_kind:', ds.output_kind)
print('cloud_region:', cat.get_daac(ds.provider).cloud_region)

### A vector example

ICESat-2 `ATL08` land/vegetation heights resolve to `output_kind == 'vector'` — point/profile data rather than a gridded raster.

In [ ]:
print(cat.get_dataset('ATL08_006').output_kind)